# Phase 2 — Exploratory Data Analysis
## SIIM-ISIC 2020 Melanoma Classification

**Purpose:** Understand class balance, metadata distributions, and image characteristics before modelling.
All figures saved to `reports/eda/`.


In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import random

plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})
sns.set_theme(style='whitegrid', palette='muted')

EDA_DIR = Path('../reports/eda')
EDA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = Path('../data/raw/isic2020/train.csv')
IMAGES_DIR = Path('../data/raw/isic2020/images')

df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df):,} rows')
print(df.dtypes)
df.head()

## 1. Class Balance

In [ ]:
vc = df['target'].value_counts()
total = len(df)
mal_pct = 100 * vc.get(1, 0) / total

print(f'Benign:    {vc.get(0, 0):6,} ({100 - mal_pct:.1f}%)')
print(f'Malignant: {vc.get(1, 0):6,} ({mal_pct:.1f}%)')
print(f'Imbalance ratio: {vc.get(0,0)/vc.get(1,1):.1f}:1')

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Benign', 'Malignant'], [vc.get(0,0), vc.get(1,0)],
              color=['#4F83CC', '#E05C5C'], edgecolor='white', width=0.5)
for bar, cnt in zip(bars, [vc.get(0,0), vc.get(1,0)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{cnt:,}', ha='center', va='bottom', fontweight='bold')
ax.set_title('Class Distribution — ISIC 2020', fontsize=13)
ax.set_ylabel('Number of Images')
plt.tight_layout()
fig.savefig(EDA_DIR / 'class_balance.png', bbox_inches='tight')
plt.show()
print(f'\n⚠️  Severe imbalance ({mal_pct:.1f}% malignant) — AUROC/recall are the right metrics, not accuracy.')

## 2. Age Distribution

In [ ]:
print(f"Missing age: {df['age_approx'].isna().sum()} ({100*df['age_approx'].isna().mean():.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall
axes[0].hist(df['age_approx'].dropna(), bins=30, color='#4F83CC', edgecolor='white', alpha=0.8)
axes[0].set_title('Age Distribution (all)')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')

# By class
for cls, color, label in [(0, '#4F83CC', 'Benign'), (1, '#E05C5C', 'Malignant')]:
    axes[1].hist(df.loc[df['target']==cls, 'age_approx'].dropna(),
                bins=25, alpha=0.6, color=color, label=label, edgecolor='white')
axes[1].set_title('Age Distribution by Class')
axes[1].set_xlabel('Age (years)')
axes[1].legend()

plt.tight_layout()
fig.savefig(EDA_DIR / 'age_distribution.png', bbox_inches='tight')
plt.show()

print('\nMedian age by class:')
print(df.groupby('target')['age_approx'].median())

## 3. Sex & Anatomical Site Distributions

In [ ]:
print(f"Missing sex:  {df['sex'].isna().sum()}")
print(f"Missing site: {df['anatom_site_general_challenge'].isna().sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Sex
sex_counts = df['sex'].fillna('unknown').value_counts()
axes[0].bar(sex_counts.index, sex_counts.values, color=['#4F83CC','#E05C5C','#7AC67A'],
            edgecolor='white')
axes[0].set_title('Sex Distribution')
axes[0].set_ylabel('Count')

# Site
site_counts = df['anatom_site_general_challenge'].fillna('unknown').value_counts()
axes[1].barh(site_counts.index, site_counts.values, color='#4F83CC', edgecolor='white')
axes[1].set_title('Anatomical Site Distribution')
axes[1].set_xlabel('Count')

plt.tight_layout()
fig.savefig(EDA_DIR / 'sex_site_distribution.png', bbox_inches='tight')
plt.show()

## 4. Malignancy Rate by Age Bucket — Core motivation for multi-modal approach

In [ ]:
df['age_bucket'] = pd.cut(df['age_approx'], bins=[0,30,50,70,100],
                           labels=['0-30','30-50','50-70','70+'])
mal_by_age = df.groupby('age_bucket', observed=True)['target'].agg(['mean','count'])
mal_by_age.columns = ['malignancy_rate', 'n']
print(mal_by_age)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(mal_by_age.index, mal_by_age['malignancy_rate'] * 100,
              color='#4F83CC', edgecolor='white')
for bar, (_, row) in zip(bars, mal_by_age.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{row['malignancy_rate']*100:.1f}%\n(n={row['n']:,})",
            ha='center', va='bottom', fontsize=9)
ax.set_title('Malignancy Rate by Age Group', fontsize=13)
ax.set_xlabel('Age Group')
ax.set_ylabel('Malignancy Rate (%)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}%'))
plt.tight_layout()
fig.savefig(EDA_DIR / 'age_vs_malignancy.png', bbox_inches='tight')
plt.show()
print('\n→ Monotonically increasing malignancy rate with age: empirical justification for multi-modal approach.')

## 5. Malignancy Rate by Anatomical Site

In [ ]:
mal_by_site = (df.groupby('anatom_site_general_challenge')['target']
                 .agg(['mean','count'])
                 .rename(columns={'mean':'malignancy_rate','count':'n'})
                 .sort_values('malignancy_rate', ascending=False))
print(mal_by_site)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(mal_by_site.index, mal_by_site['malignancy_rate'] * 100,
              color='#5CAE73', edgecolor='white')
ax.set_title('Malignancy Rate by Anatomical Site', fontsize=13)
ax.set_xlabel('Anatomical Site')
ax.set_ylabel('Malignancy Rate (%)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}%'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig.savefig(EDA_DIR / 'site_vs_malignancy.png', bbox_inches='tight')
plt.show()

## 6. Patient-Level Leakage Check

In [ ]:
if 'patient_id' in df.columns:
    imgs_per_patient = df.groupby('patient_id').size()
    multi = (imgs_per_patient > 1).sum()
    print(f'Total unique patients: {imgs_per_patient.nunique():,}')
    print(f'Patients with >1 image: {multi:,} ({100*multi/len(imgs_per_patient):.1f}%)')
    print(f'Max images per patient: {imgs_per_patient.max()}')
    print('\n⚠️ Patient-level split is mandatory — splitting by image would cause data leakage!')
else:
    print('No patient_id column found — check CSV structure.')

## 7. Sample Images — Benign vs Malignant

In [ ]:
def load_img(name):
    for ext in ['.jpg', '.jpeg', '']:
        p = IMAGES_DIR / f'{name}{ext}'
        if p.exists():
            return Image.open(p).convert('RGB')
    return None

n_each = 4
benign_sample = df[df['target']==0].sample(n_each, random_state=42)['image_name'].tolist()
malignant_sample = df[df['target']==1].sample(n_each, random_state=42)['image_name'].tolist()

fig, axes = plt.subplots(2, n_each, figsize=(14, 7))
for i, name in enumerate(benign_sample):
    img = load_img(name)
    if img:
        axes[0, i].imshow(img)
    axes[0, i].set_title(f'Benign\n{name[:12]}', fontsize=8)
    axes[0, i].axis('off')

for i, name in enumerate(malignant_sample):
    img = load_img(name)
    if img:
        axes[1, i].imshow(img)
    axes[1, i].set_title(f'Malignant\n{name[:12]}', fontsize=8, color='#c62828')
    axes[1, i].axis('off')

fig.suptitle('Sample Dermoscopy Images — Benign vs Malignant', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(EDA_DIR / 'sample_images.png', bbox_inches='tight')
plt.show()

## 8. Summary

In [ ]:
print('=== EDA Summary ===')
print(f'Total images:       {len(df):,}')
print(f'Malignant:          {df["target"].sum():,} ({100*df["target"].mean():.1f}%)')
print(f'Missing age:        {df["age_approx"].isna().sum():,}')
print(f'Missing sex:        {df["sex"].isna().sum():,}')
print(f'Missing site:       {df["anatom_site_general_challenge"].isna().sum():,}')
print(f'\nAll EDA figures saved to: reports/eda/')
print('\nNext step: run src/data/preprocess.py to create patient-level splits.')